In [0]:
--------Ejercicio 1: El "Ritmo de Carrera" (Promedio de vueltas limpias)-------------
SELECT
    dri.full_name,
    dri.team_name,
    ROUND(AVG(lap.lap_duration), 2) AS avg_lap_duration
FROM
    workspace.formula_1.bronze_laps lap
JOIN 
    workspace.formula_1.bronze_sessions ses ON lap.session_key = ses.session_key
JOIN 
    workspace.formula_1.bronze_meetings met ON ses.meeting_key = met.meeting_key
JOIN
    workspace.formula_1.bronze_drivers dri ON lap.driver_number = dri.driver_number AND met.meeting_key = dri.meeting_key AND ses.session_key = dri.session_key
WHERE
    ses.session_type = 'Race'
    AND
    lap.is_pit_out_lap = false
    AND 
    met.country_name = 'Monaco'
GROUP BY
    dri.full_name,
    dri.team_name
ORDER BY
    AVG(lap.lap_duration) ASC
;


--------Ejercicio 2: Consistencia entre Sectores--------
SELECT
    dri.full_name,
    dri.team_name,
    COUNT(lap.lap_number) AS total_laps,
    ROUND(STDDEV(lap.duration_sector_1), 3) AS std_dev_sector_1,
    ROUND(STDDEV(lap.duration_sector_2), 3) AS std_dev_sector_2
FROM
    workspace.formula_1.bronze_laps lap
JOIN 
    workspace.formula_1.bronze_sessions ses ON lap.session_key = ses.session_key
JOIN 
    workspace.formula_1.bronze_meetings met ON ses.meeting_key = met.meeting_key
JOIN
    workspace.formula_1.bronze_drivers dri ON lap.driver_number = dri.driver_number AND met.meeting_key = dri.meeting_key AND ses.session_key = dri.session_key
WHERE
    lap.is_pit_out_lap = false
    AND 
    ses.session_key = 9686
GROUP BY
    dri.full_name,
    dri.team_name
HAVING
    total_laps > 10
ORDER BY
    4, 5 ASC
;


---------------Ejercicio 3: Evolución de las posiciones en "Tiempo Real"-----------------
WITH cumulative_lap_duration AS (
    SELECT
        lap.lap_number,
        dri.full_name,
        dri.team_name,
        SUM(lap.lap_duration) OVER (PARTITION BY ses.session_key, dri.driver_number ORDER BY lap.lap_number) AS race_accumulated_time
    FROM
        workspace.formula_1.bronze_laps lap
    JOIN 
        workspace.formula_1.bronze_sessions ses ON lap.session_key = ses.session_key
    JOIN 
        workspace.formula_1.bronze_meetings met ON ses.meeting_key = met.meeting_key
    JOIN
        workspace.formula_1.bronze_drivers dri ON lap.driver_number = dri.driver_number AND met.meeting_key = dri.meeting_key AND ses.session_key = dri.session_key
    WHERE
        lap.lap_duration IS NOT NULL
        AND 
        ses.session_key = 9686
),
driver_ranking AS (
    SELECT
        *,
        DENSE_RANK() OVER(PARTITION BY lap_number ORDER BY race_accumulated_time ASC) AS driver_rank
    FROM
        cumulative_lap_duration
)
SELECT
    *
FROM
    driver_ranking
ORDER BY 
    lap_number ASC, driver_rank ASC
;

---------Ejercicio 4: Análisis de la "Vuelta Rápida" del Gran Premio-------------
WITH drivers_ranking AS (
    SELECT
        -- met.year,
        -- met.month,
        met.circuit_short_name,
        dri.full_name,
        dri.team_name,
        lap.lap_number,
        lap.lap_duration,
        ROW_NUMBER() OVER(PARTITION BY met.circuit_short_name ORDER BY lap.lap_duration ASC) AS row_num
    FROM
        workspace.formula_1.bronze_laps lap
    JOIN 
        workspace.formula_1.bronze_sessions ses ON lap.session_key = ses.session_key
    JOIN 
        workspace.formula_1.bronze_meetings met ON ses.meeting_key = met.meeting_key
    JOIN
        workspace.formula_1.bronze_drivers dri ON lap.driver_number = dri.driver_number AND met.meeting_key = dri.meeting_key AND ses.session_key = dri.session_key
    WHERE
        lap.is_pit_out_lap = false
        AND 
        lap.lap_duration IS NOT NULL
)
SELECT
    -- year,
    -- month,
    circuit_short_name,
    full_name,
    team_name,
    lap_number,
    lap_duration
FROM
    drivers_ranking
WHERE
    row_num = 1
ORDER BY
    circuit_short_name,
    lap_duration ASC
;

------------Ejercicio 5: Detección de "Anomalías" en Carrera (Safety Car / Incidentes)------------
WITH lap_driver_anomaly AS 
(
    SELECT
        dri.full_name,
        lap.lap_number,
        lap.duration_sector_3,
        AVG(lap.duration_sector_3) OVER(PARTITION BY ses.session_key, dri.driver_number) AS promedio
    FROM
        workspace.formula_1.bronze_laps lap
    JOIN 
        workspace.formula_1.bronze_sessions ses ON lap.session_key = ses.session_key
    JOIN 
        workspace.formula_1.bronze_meetings met ON ses.meeting_key = met.meeting_key
    JOIN
        workspace.formula_1.bronze_drivers dri ON lap.driver_number = dri.driver_number AND met.meeting_key = dri.meeting_key AND ses.session_key = dri.session_key
    WHERE
        lap.is_pit_out_lap = false
        AND 
        lap.duration_sector_3 IS NOT NULL
        AND
        ses.session_key = 9686
)
SELECT
    full_name,
    lap_number,
    duration_sector_3,
    ROUND(promedio, 3) AS driver_avg_sector_3
FROM
    lap_driver_anomaly
WHERE
    duration_sector_3 >= promedio * 1.5
ORDER BY
    full_name ASC,
    lap_number ASC
;

------------Ejercicio 6: El "Piloto Ideal" (Vuelta Teórica Perfecta)------------
WITH ideal_pilot AS (
    SELECT
        dri.full_name,
        dri.team_name,
        MIN(lap.duration_sector_1) + MIN(lap.duration_sector_2) + MIN(lap.duration_sector_3) AS theoretical_best_lap,
        MIN(lap.lap_duration) AS actual_best_lap
    FROM
        workspace.formula_1.bronze_laps lap
    JOIN 
        workspace.formula_1.bronze_sessions ses ON lap.session_key = ses.session_key
    JOIN 
        workspace.formula_1.bronze_meetings met ON ses.meeting_key = met.meeting_key
    JOIN
        workspace.formula_1.bronze_drivers dri ON lap.driver_number = dri.driver_number AND met.meeting_key = dri.meeting_key AND ses.session_key = dri.session_key
    WHERE
        lap.is_pit_out_lap = false
        AND 
        lap.duration_sector_1 IS NOT NULL
        AND 
        lap.duration_sector_2 IS NOT NULL
        AND 
        lap.duration_sector_3 IS NOT NULL
        AND 
        lap.lap_duration IS NOT NULL
        AND
        ses.session_key = 9686
    GROUP BY
        dri.full_name,
        dri.team_name
)
SELECT
    full_name,
    team_name,
    ROUND(theoretical_best_lap, 3) AS theoretical_best_lap,
    actual_best_lap,
    ROUND(actual_best_lap - theoretical_best_lap, 3) AS time_gap
FROM
    ideal_pilot
ORDER BY
    time_gap ASC;